<a href="https://colab.research.google.com/github/nikunj555/My-Projects/blob/main/Malicious_URL_Detection_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import re
from urllib.parse import urlparse
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.read_csv(
    "benign_vs_defacement_urls.csv",
    encoding="latin1",
    on_bad_lines="skip"
)

df.columns = df.columns.str.strip().str.lower()

if "label" not in df.columns:
    for col in df.columns:
        if col != "url":
            df = df.rename(columns={col: "label"})
            break

df = df.dropna(subset=["url", "label"])

df["label"] = df["label"].astype(str).str.lower()

df["label"] = df["label"].replace({
    "benign": 0,
    "good": 0,
    "0": 0,
    0: 0,
    "defacement": 1,
    "bad": 1,
    "1": 1,
    1: 1
})

df = df[df["label"].isin([0, 1])]

def url_length(url):
    return len(url)

def hostname_length(url):
    try:
        return len(urlparse(url).netloc)
    except:
        return 0

def count_symbols(url):
    return [
        url.count("."),
        url.count("-"),
        url.count("@"),
        url.count("?"),
        url.count("%"),
        url.count("=")
    ]

def count_digits(url):
    return sum(c.isdigit() for c in url)

def has_ip(url):
    return 1 if re.search(r"\d+\.\d+\.\d+\.\d+", url) else 0

def is_shortened(url):
    return 1 if any(s in url for s in ["bit.ly", "tinyurl", "goo.gl", "t.co"]) else 0

def count_slashes(url):
    return url.count("/")

df["url_len"] = df["url"].apply(url_length)
df["host_len"] = df["url"].apply(hostname_length)

df[["dot","dash","at","question","percent","equal"]] = pd.DataFrame(
    df["url"].apply(count_symbols).tolist(),
    index=df.index
)

df["digits"] = df["url"].apply(count_digits)
df["has_ip"] = df["url"].apply(has_ip)
df["short_url"] = df["url"].apply(is_shortened)
df["slashes"] = df["url"].apply(count_slashes)

good = df[df["label"] == 0]
bad = df[df["label"] == 1]

good_downsampled = resample(
    good,
    replace=False,
    n_samples=len(bad),
    random_state=42
)

df_balanced = pd.concat([good_downsampled, bad])

X = df_balanced.drop(columns=["url", "label"])
y = df_balanced["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Confusion Matrix:")
print(cm)

/tmp/ipykernel_7604/1286036899.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["label"] = df["label"].replace({


Accuracy: 0.9709945825449079
Confusion Matrix:
[[18621   680]
 [  439 18839]]
